## KST Transformer - Izveštaj

### Cilj projekta

Cilj projekta je treniranje transformer modela koji replicira **IITA (Inductive Item Tree Analysis)** algoritam iz **Knowledge Space Theory (KST)**.

IITA algoritam rekonstruiše **prerequisit relacije** između pitanja na osnovu matrice odgovora studenata. Ove relacije opisuju koji koncepti moraju biti usvojeni pre nego što student može uspešno odgovoriti na određeno pitanje.

Transformer model uči da iz iste matrice odgovora direktno predvidi **adjacency matricu prerequisita**, bez eksplicitnog pokretanja IITA algoritma - što je računski znatno efikasnije.

---

### Osnovni koncepti

- **Pitanje (item)**: zadatak koji student može rešiti ili ne
- **Prerequisit relacija** `i -> j`: pitanje `i` je prerequisit pitanja `j` - student koji zna `j` mora znati i `i`
- **Quasi-order**: tranzitivna i refleksivna relacija prerequisita

### 1. Generisanje dataseta

**Fajl:** `scripts/generate_dataset.py`

Dataset se sintetički generiše koristeći `learning_spaces` biblioteku. Svaki primer (test) sadrži:
- `X` - binarna matrica odgovora studenata oblika `(students, items)` (1 -> tačan odgovor, 0 -> netačan odgovor)
- `Y` - adjacency matrica prerequisita oblika `(items, items)` - bez tranzitivnog zatvorenja (ukoliko `a->b` i `b->c`, matrica ne sadrzi i `a->c`)
- `item_count` - stvarni broj pitanja u primeru (ostalo je padding nulama)

### Pipeline generisanja jednog primera

1. Nasumično se bira broj pitanja `n` iz `[min_items, max_items]` (broj pitanja se bira uniformno ili po linearnim tezinama)
2. Generišu se nasumične implikacije kao DAG (topološka permutacija garantuje da relacije mogu ići u bilo kom smeru)
3. Računa se **tranzitivno zatvorenje** nad skupom implikacija - šalje se `simu()` funkciji
4. `simu()` simulira odgovore uz šum: `ce` (careless error) i `lg` (lucky guess)
5. Računa se **tranzitivna redukcija** - čuva se kao `Y`
6. Response matrica i adj matrica se padduju na `max_items`

### Topološka permutacija

Bez permutacije, implikacije bi uvek imale oblik `i -> j` gde je `i < j` (gornji trougao). Na taj način bi uveli određeni šablon na izlazu koji bi model mogao da nauči. Permutacija to sprečava:

```python
topo     = np.random.permutation(items)
possible = [(int(topo[i]), int(topo[j])) for i in range(items) for j in range(i+1, items)]
```

### Parametri šuma

| Parametar | Vrednost | Opis |
|-----------|----------|------|
| `ce` | 0.10 | Careless error - student koji zna odgovor ipak greši |
| `lg` | 0.05 | Lucky guess - student koji ne zna ipak odgovori tačno |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.load('data/kst_dataset_weighted.npz')
X, Y, item_counts = data['X'], data['Y'], data['item_counts']

print(f'Broj uzoraka:      {X.shape[0]}')
print(f'Studenata/uzorak:  {X.shape[1]}')
print(f'Max items:         {X.shape[2]}')
print()
print('Distribucija broja pitanja po uzorku:')
for k in range(2, 6):
    cnt = (item_counts == k).sum()
    print(f'  {k} pitanja: {cnt:5d} uzoraka ({100*cnt/len(item_counts):.1f}%)')

Broj uzoraka:      1000
Studenata/uzorak:  200
Max items:         5

Distribucija broja pitanja po uzorku:
  2 pitanja:   262 uzoraka (26.2%)
  3 pitanja:   224 uzoraka (22.4%)
  4 pitanja:   257 uzoraka (25.7%)
  5 pitanja:   257 uzoraka (25.7%)


In [ ]:
idx = 0
n   = item_counts[idx]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(X[idx, :20, :n], cmap='Blues', vmin=0, vmax=1, aspect='auto')
axes[0].set_title(f'Response matrica (20/{X.shape[1]} studenata, {n} pitanja)')
axes[0].set_xlabel('Pitanje')
axes[0].set_ylabel('Student')
axes[0].set_xticks(range(n))

axes[1].imshow(Y[idx, :n, :n], cmap='Reds', vmin=0, vmax=1)
axes[1].set_title('Adjacency matrica prerequisita')
axes[1].set_xlabel('j (zahteva prerequisit)')
axes[1].set_ylabel('i (prerequisit)')
axes[1].set_xticks(range(n))
axes[1].set_yticks(range(n))
for i in range(n):
    for j in range(n):
        axes[1].text(j, i, str(int(Y[idx, i, j])), ha='center', va='center', fontsize=14)

plt.tight_layout()
plt.show()
print(f'item_count = {n}')

### 2. Dataset klasa

**Fajl:** `src/kst/dataset.py`

### KSTDataset

PyTorch `Dataset` koji učitava `.npz` fajl i vraća trojku `(X, Y, item_count)` po uzorku.

### make_loss_mask

Kreira boolean masku oblika `(batch, max_items, max_items)` - `True` tamo gde treba računati gubitak. Maskiraju se padding ćelije i dijagonala:

```python
idx        = torch.arange(max_items)
valid      = idx.unsqueeze(0) < item_counts.unsqueeze(1)       # (batch, max_items)
valid_pair = valid.unsqueeze(2) & valid.unsqueeze(1)           # (batch, max_items, max_items)
diag_mask  = ~torch.eye(max_items, dtype=torch.bool).unsqueeze(0)
return valid_pair & diag_mask
```

In [ ]:
import sys
sys.path.insert(0, 'src')

import torch
from kst.dataset import make_dataloaders, make_loss_mask

train_loader, val_loader, test_loader = make_dataloaders(
    'data/kst_dataset.npz',
    batch_size=64, val_ratio=0.2, test_ratio=0.1, seed=42,
)

print(f'Train batches: {len(train_loader)}')
print(f'Val   batches: {len(val_loader)}')
print(f'Test  batches: {len(test_loader)}')

X_b, Y_b, ic_b = next(iter(train_loader))
print(f'\nX oblik:         {X_b.shape}   (batch, students, items)')
print(f'Y oblik:         {Y_b.shape}   (batch, items, items)')
print(f'item_counts[:5]: {ic_b[:5].tolist()}')

mask = make_loss_mask(ic_b, max_items=X_b.shape[-1])
print(f'\nLoss maska primer [0] (item_count={ic_b[0]}):')
print(mask[0].int())

### 3. Arhitektura modela

**Fajl:** `src/kst/model.py`

Model je **encoder-only transformer** koji procesira pitanja kao tokene i za svaki par predviđa prerequisit relaciju.

```
Ulaz: (batch, students, items)
    |
    v  Transpose
(batch, items, students)
    |
    v  Input Projection: Linear(students -> d_model)
(batch, items, d_model)
    |
    v  + Learned Positional Encoding: Embedding(max_items, d_model)
    |
    v  Transformer Encoder [sa padding maskom]
h: (batch, items, d_model)
    |
    v  Pair Classifier
       konkatenira h_i i h_j za svaki par (i, j)
       Linear(2*d_model -> d_model) -> ReLU -> Dropout -> Linear(d_model -> 1)
    |
    v
Izlaz: (batch, items, items) - logiti
```

**Padding maska**: `src_key_padding_mask` u PyTorch TransformerEncoder-u - `True` = ignoriši tu poziciju u attention-u. Sprečava model da vidi paddovana pitanja.

**Logiti**: Model vraća logite (bez sigmoid). `BCEWithLogitsLoss` interno primenjuje sigmoid za numeričku stabilnost.

In [ ]:
from kst.model import KSTTransformer

# Optimalni hiperparametri pronadjeni random search-om
model = KSTTransformer(
    max_items=5, students=500,
    d_model=64, nhead=8, num_encoder_layers=4,
    dim_feedforward=512, dropout=0.1,
)

total = sum(p.numel() for p in model.parameters())
print(f'Ukupno parametara: {total:,}\n')
print(model)

In [ ]:
x_test  = torch.randint(0, 2, (4, 500, 5)).float()
ic_test = torch.tensor([5, 4, 3, 2])
out     = model(x_test, ic_test)
print(f'Ulaz:  {x_test.shape}  (batch=4, students=500, max_items=5)')
print(f'Izlaz: {out.shape}   (batch=4, items=5, items=5) - logiti')

### 4. Random Search hiperparametara

**Fajl:** `scripts/random_search.py`

Umesto ručnog podešavanja hiperparametara, korišćen je **random search**: nasumično se uzorkuju kombinacije hiperparametara, svaka se trenira ograničen broj epoha, i bira se kombinacija sa najnižim `val_loss`.

### Prostor pretrage

| Hiperparametar | Opseg |
|----------------|-------|
| `d_model` | {64, 128, 256} |
| `nhead` | {2, 4, 8} |
| `num_layers` | {2, 3, 4} |
| `dim_feedforward` | {128, 256, 512} |
| `dropout` | {0.1, 0.2, 0.3} |
| `batch_size` | {32, 64, 128} |
| `lr` | log-uniform [1e-4, 1e-3] |

Pokrenuto je **50 trial-ova**, svaki treniran najviše 30 epoha sa early stopping patience=10.

In [ ]:
import pandas as pd

df = pd.read_csv('report material/random_search.csv')
df = df.sort_values('val_loss').reset_index(drop=True)
df.index += 1

print('Top 10 kombinacija (sortirano po val_loss):')
print(df[['val_loss','d_model','nhead','num_layers','dim_feedforward','dropout','batch_size','lr']].head(10).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].scatter(range(1, len(df)+1), df['val_loss'], color='steelblue', s=40)
axes[0].scatter([1], [df['val_loss'].iloc[0]], color='tomato', s=100, zorder=5, label='Najbolja kombinacija')
axes[0].set_title('Val loss po trial-u (sortirano)')
axes[0].set_xlabel('Rang')
axes[0].set_ylabel('Val loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

param_counts = df.groupby('num_layers')['val_loss'].mean().sort_index()
axes[1].bar(param_counts.index.astype(str), param_counts.values, color='steelblue', alpha=0.8)
axes[1].set_title('Prosecni val loss po broju slojeva')
axes[1].set_xlabel('num_layers')
axes[1].set_ylabel('Prosecni val loss')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

#### Izabrani hiperparametri

| Parametar | Vrednost |
|-----------|----------|
| `d_model` | 64 |
| `nhead` | 8 |
| `num_layers` | 4 |
| `dim_feedforward` | 512 |
| `dropout` | 0.1 |
| `batch_size` | 64 |
| `lr` | 0.000185 |

### 5. Trening

**Fajl:** `scripts/train.py`

### Loss funkcija

**Binary Cross-Entropy with Logits** sa `pos_weight` parametrom koji kompenzuje neravnotežu klasa (~82% nula, ~18% jedinica):

$$\text{pos\_weight} = \frac{\text{broj nula}}{\text{broj jedinica}} \approx 4.6$$

Loss se računa samo na validnim ćelijama (bez paddinga i dijagonale), uz pomoć `make_loss_mask`.

### Evaluacione metrike

| Metrika | Formula | Opis |
|---------|---------|------|
| **F1** | $2TP / (2TP + FP + FN)$ | Harmonijska sredina precision i recall |
| **Hamming loss** | $(FP + FN) / \text{ukupno}$ | Udeo pogrešno klasifikovanih celija |

Obe metrike se prosecavaju po uzorcima (makro prosek). Uzorci bez prerequisita dobijaju F1 = 1.0 ako je predikcija ispravno prazna.

![](report%20material/training_curves.png)

### 6. Evaluacija - KST Transformer vs IITA

**Fajl:** `scripts/eval_iita.py`

Poređenje KST Transformer modela sa IITA algoritmom (v=1, `iita_exclude_transitive`) na istom test skupu.

**Metodologija:**
- Isti test split kao u treningu (seed=42, test_ratio=0.1)
- Oba metoda dobijaju iste ulazne podatke
- IITA se pokreće uzorak po uzorak samo na stvarnim pitanjima (bez paddinga)
- Metrike se računaju po uzorku i usrednjavaju (makro prosek)
- Uzorci bez prerequisita: F1 = 1.0 ako model ispravno predvidi sve nule

In [ ]:
import numpy as np
import torch
from kst.dataset import make_dataloaders
from kst.model import KSTTransformer
from learning_spaces.kst.iita import iita_exclude_transitive

def metrics_np(pred_adj, true_adj, n_items):
    mask      = ~np.eye(n_items, dtype=bool)
    pred_flat = pred_adj[mask].astype(bool)
    true_flat = true_adj[mask].astype(bool)
    tp = ( pred_flat &  true_flat).sum()
    fp = ( pred_flat & ~true_flat).sum()
    fn = (~pred_flat &  true_flat).sum()
    f1      = (2*tp)/(2*tp+fp+fn) if (tp+fp+fn) > 0 else 1.0
    hamming = (pred_flat != true_flat).sum() / len(pred_flat)
    return float(f1), float(hamming)

_, _, test_loader = make_dataloaders('data/kst_dataset.npz', batch_size=64,
                                      val_ratio=0.2, test_ratio=0.1, seed=42)
X_all  = torch.cat([x  for x, _, _  in test_loader])
Y_all  = torch.cat([y  for _, y, _  in test_loader])
ic_all = torch.cat([ic for _, _, ic in test_loader])

NUM_SAMPLES = 500
X_sub, Y_sub, ic_sub = X_all[:NUM_SAMPLES], Y_all[:NUM_SAMPLES], ic_all[:NUM_SAMPLES]
_, students, max_items = X_sub.shape

# --- IITA ---
print(f'Pokrecem IITA na {NUM_SAMPLES} uzoraka...')
iita_f1s, iita_ham = [], []
for i in range(NUM_SAMPLES):
    n = int(ic_sub[i])
    try:
        res  = iita_exclude_transitive(X_sub[i, :, :n].numpy(), v=1)
        pred = np.zeros((n, n), dtype=np.float32)
        for (a, b) in res['implications']:
            if a < n and b < n:
                pred[a][b] = 1.0
    except:
        continue
    f1, h = metrics_np(pred, Y_sub[i, :n, :n].numpy(), n)
    iita_f1s.append(f1)
    iita_ham.append(h)

# --- Transformer ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt   = torch.load('checkpoints/best.pt', map_location=device)
sa     = ckpt.get('args', {})
model  = KSTTransformer(
    max_items=max_items, students=students,
    d_model=sa.get('d_model', 64), nhead=sa.get('nhead', 8),
    num_encoder_layers=sa.get('num_layers', 4),
    dim_feedforward=sa.get('dim_feedforward', 512),
    dropout=sa.get('dropout', 0.1),
).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

tr_f1s, tr_ham = [], []
with torch.no_grad():
    preds = model(X_sub.to(device), ic_sub.to(device)).cpu().numpy()
for i in range(NUM_SAMPLES):
    n    = int(ic_sub[i])
    pred = (1 / (1 + np.exp(-preds[i, :n, :n])) > 0.5).astype(np.float32)
    f1, h = metrics_np(pred, Y_sub[i, :n, :n].numpy(), n)
    tr_f1s.append(f1)
    tr_ham.append(h)

print(f'\n{"Metod":<22} {"F1":>8} {"Hamming loss":>14}')
print('-' * 46)
print(f'{"IITA (v=1)":<22} {np.mean(iita_f1s):>8.3f} {np.mean(iita_ham):>14.3f}')
print(f'{"KST Transformer":<22} {np.mean(tr_f1s):>8.3f} {np.mean(tr_ham):>14.3f}')
print(f'\nCheckpoint: epoha {ckpt.get("epoch","?")} | val_loss={ckpt.get("val_loss",0):.4f} | val_F1={ckpt.get("val_f1",0):.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(iita_f1s, bins=20, alpha=0.7, label='IITA', color='steelblue')
axes[0].hist(tr_f1s,   bins=20, alpha=0.7, label='Transformer', color='tomato')
axes[0].set_title('Distribucija F1 po uzorku')
axes[0].set_xlabel('F1')
axes[0].set_ylabel('Broj uzoraka')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(iita_ham, bins=20, alpha=0.7, label='IITA', color='steelblue')
axes[1].hist(tr_ham,   bins=20, alpha=0.7, label='Transformer', color='tomato')
axes[1].set_title('Distribucija Hamming loss po uzorku')
axes[1].set_xlabel('Hamming loss')
axes[1].set_ylabel('Broj uzoraka')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Rezultati

| Metod | F1 | Hamming loss |
|-------|----|--------------| 
| IITA (v=1) | 0.807 | 0.060 |
| **KST Transformer** | **0.900** | **0.049** |